# D02-质量检查与清洗

D01 看清了字段与时间覆盖，这一本回答报告的核心问题：**这份数据能不能直接用来做取消预测？**

做法是固定的四步：提出假设 → 用数据验证 → 量化证据 → 登记决定。
16 项检查覆盖七个维度（完整性、规范性、唯一性、一致性、准确性、时效性、代表性），
结论落在 `dq_check_result` 表里，供报告与图表引用。

## 16 项检查从哪里来

| 维度 | 检查项 | 编号 |
| --- | --- | --- |
| 完整性 | 整列高缺失、关键字段缺失 | DQ-02、DQ-03 |
| 规范性 | 数值字段类型、日期与月份编码 | DQ-04、DQ-05 |
| 唯一性 | 完全重复、主键唯一性 | DQ-06、DQ-07 |
| 一致性 | 状态码与取消标记、状态日期时序 | DQ-08、DQ-09 |
| 准确性 | 房价异常、无人入住与零晚数、提前期极值、字典可解释性 | DQ-10—DQ-12、DQ-16 |
| 时效性 | 到达日期覆盖 | DQ-01 |
| 代表性 | 时段、客源与酒店、圈选口径 | DQ-13—DQ-15 |

判定阈值全部在 `configs/settings.yaml`，代码里不写 0.5、0.1 这种数字。

### 环境准备

In [ ]:
# ── 第一格：环境准备（每本 notebook 开头都是这十几行）──
%load_ext autoreload
%autoreload 2
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hotel_dq import config, db, quality, viz
from hotel_dq.viz import MAIN, MAIN_LIGHT, MAIN_MID, MAIN_DARK, GRAY, GRAY_DARK, RED, AMBER, INK, MUTED, FAINT   # 配色常量

viz.apply_theme()                              # 字体、配色、坐标轴样式：一行
con = db.connect()                             # 路径从包里来：notebook 在哪打开都找得到库
NB = "D02_质量检查与清洗"
SOURCE = f"{config.SOURCE_NOTE}；脚本：{NB}"     # 每张图左下角的来源行
pd.set_option("display.max_columns", 20)

In [ ]:
# 自检：版本、中文字体、库大小——环境不对，后面的图会全变方框
import sys, matplotlib
from matplotlib import font_manager

installed = {f.name for f in font_manager.fontManager.ttflist}
cn_font = [f for f in plt.rcParams["font.family"] if f in installed and f != "DejaVu Sans"]

print(f"Python      {sys.version.split()[0]}")
print(f"pandas      {pd.__version__}")
print(f"matplotlib  {matplotlib.__version__}")
print(f"SQLite      {sqlite3.sqlite_version}")
print(f"中文字体    {cn_font}")
print(f"数据库      {config.DB_PATH.name}（{config.DB_PATH.stat().st_size / 1e6:.0f} MB）")

## 跑 16 项检查

In [ ]:
from hotel_dq import dwd

raw = pd.read_sql(f"SELECT * FROM {config.TABLE}", con)
enriched = dwd.with_derived(raw)                 # 补齐派生列；已有就不动
settings = config.load_settings()
assert len(raw) == 119_390
print(f"原始 {len(raw):,} 行 × {len(raw.columns)} 列；补上分区列后 {len(enriched.columns)} 列；参数版本 {settings['version']}")

In [ ]:
from hotel_dq import checks

res = checks.run_all(enriched, settings)

# 对账：16 项齐全，状态分布必须是报告里的 3 / 9 / 4
assert len(res) == 16
counts = res["状态"].value_counts().to_dict()
assert counts == {"警告": 9, "不通过": 4, "通过": 3}, counts
print(f"16 项检查：通过 {counts['通过']}、警告 {counts['警告']}、不通过 {counts['不通过']}")

res[["check_id", "维度", "检查项", "结果", "状态"]]

In [ ]:
tab = (res.pivot_table(index="维度", columns="状态", values="check_id", aggfunc="count", fill_value=0)
          .reindex(columns=["不通过", "警告", "通过"], fill_value=0))
tab = tab.loc[tab.sum(axis=1).sort_values().index]        # 问题多的维度放上面
tab

结论一眼能看：**3 项通过、9 项警告、4 项不通过**。不通过的是 `company_id` 缺失 94%、
完全重复 27%、房价有 0 值与极值、样本只有 2 家酒店——这四项都不是"删空值"能解决的。

### 图 D02_01：各维度的检查结果

In [ ]:
# ── 出图 D02_01：各维度的检查结果 ──────────────────────────────────────
# 为什么用堆叠横条：三种状态互斥且穷尽，加起来就是该维度的项数，堆叠能同时看总量与构成。
# 为什么每段写字：只靠颜色区分状态，色弱读者读不出来（规范要求"标注直接写在图上"）。
# 规范落在哪几行：横轴从 0 开始；红=不通过、黄=警告、蓝=通过；段内写"状态 项数"，不写图例。
STATUS = {"不通过": RED, "警告": AMBER, "通过": MAIN_MID}
n_pass, n_warn, n_fail = counts["通过"], counts["警告"], counts["不通过"]

fig, ax = plt.subplots(figsize=(10, 4.6))
left = np.zeros(len(tab))
for state in ["不通过", "警告", "通过"]:
    vals = tab[state].to_numpy()
    ax.barh(np.arange(len(tab)), vals, left=left, color=STATUS[state],
            height=0.58, edgecolor="white", lw=1.2)
    for i, v in enumerate(vals):
        if v:
            ax.text(left[i] + v / 2, i, f"{state} {v}", ha="center", va="center",
                    fontsize=9.5, fontweight="bold",
                    color="white" if state != "通过" else INK)
    left += vals

for i, v in enumerate(left):
    ax.text(v + 0.1, i, f"共 {int(v)} 项", va="center", fontsize=9, color=MUTED)

ax.set_yticks(np.arange(len(tab)), tab.index, fontsize=10.5)
ax.set_xlim(0, left.max() + 1.2)
ax.set_xticks([])
ax.grid(False)
ax.tick_params(length=0)

# ---- 标题、副标题 ----
title = f"16 项检查：{n_pass} 项通过、{n_warn} 项警告、{n_fail} 项不通过"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = (f"七个维度共 16 项，判定阈值来自 configs/settings.yaml（通过阈值 "
            f"{settings['quality']['pass_threshold']:.0%}）\n"
            "不通过集中在完整性、唯一性、准确性、代表性；一致性与时效性没有“不通过”项")
viz.finish(fig, title, subtitle, SOURCE, "D02_01_各维度检查结果.png")
plt.show()

In [ ]:
# ---- 读回断言：每根堆叠条的总长 = 该维度的项数 ----
# 三个状态各是一个 container，按"行"把三段的宽度加起来，才是这一根条的总长
total = np.array([sum(ax.containers[k][i].get_width() for k in range(len(tab.columns)))
                  for i in range(len(tab))])
assert np.allclose(total, tab.sum(axis=1).to_numpy()), "堆叠条总长和维度项数对不上"
assert f"{n_fail} 项不通过" in fig.texts[0].get_text(), "标题里的项数不是从结果算的"
print("读回断言通过：堆叠条总长 = 各维度项数")

## 四步清洗

清洗顺序是契约：**无人入住 → 零晚数 → 房价异常 → 完全重复**。
每行只计入第一个命中的步骤，所以四步之和等于总剔除量；顺序一换，各步数字就变。

In [ ]:
clean, steps = dwd.clean_bookings(raw, settings)
step_df = pd.DataFrame(steps)
step_df["占原始比例"] = step_df["剔除行数"] / len(raw)

# 对账：四步剔除数与报告表 5-1 完全一致
assert step_df["剔除行数"].tolist() == [180, 645, 1_167, 31_813], step_df["剔除行数"].tolist()
assert len(clean) == 85_585
print(f"四步剔除 {len(raw) - len(clean):,} 行（{quality.fmt_pct(1 - len(clean) / len(raw))}），保留 {len(clean):,} 行")

step_df

### 图 D02_02：四步清洗的剩余记录

In [ ]:
# ── 出图 D02_02：四步清洗的剩余记录数 ──────────────────────────────────
# 为什么用横条而不是漏斗图：漏斗的斜边不携带信息，还会让读者去比角度；横条的长度就是行数。
# 为什么把"原始"也画进去：不画基准，读者不知道 31,813 是多大一块。
# 规范落在哪几行：横轴从 0 开始；每步写"剩余"和"本步剔除"；主色给剔除最多的一步。
rows = [("原始记录", len(raw), 0)] + [(s["步骤"], s["剩余行数"], s["剔除行数"]) for s in steps]
names = [r[0] for r in rows]
remain_vals = np.array([r[1] for r in rows], dtype=float)
removed_vals = np.array([r[2] for r in rows], dtype=float)
top_i = int(np.argmax(removed_vals))

fig, ax = plt.subplots(figsize=(10, 4.4))
y = np.arange(len(rows))
colors = [GRAY if i == 0 else (MAIN if i == top_i else MAIN_MID) for i in range(len(rows))]
ax.barh(y, remain_vals, color=colors, height=0.58)
for i, (nm, rem, cut) in enumerate(rows):
    label = f"{int(rem):,} 行" if i == 0 else f"{int(rem):,} 行　（本步剔除 {int(cut):,}）"
    ax.text(rem + len(raw) * 0.012, i, label, va="center", fontsize=10,
            color=INK if i else MUTED, fontweight="bold" if i == top_i else "normal")

ax.set_yticks(y, names, fontsize=10.5)
ax.invert_yaxis()
ax.set_xlim(0, len(raw) * 1.34)
ax.set_xticks([])
ax.grid(False)
ax.tick_params(length=0)

removed = len(raw) - len(clean)
dup_share = steps[-1]["剔除行数"] / removed
title = f"四步剔除 {removed:,} 行（{quality.fmt_pct(removed / len(raw))}），{quality.fmt_pct(dup_share)} 来自去重"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = ("顺序固定：无人入住 → 零晚数 → 房价异常 → 完全重复；每行只计入第一个命中的步骤\n"
            "前两步合计只有 825 行，说明业务字段本身很干净；问题集中在重复与房价")
viz.finish(fig, title, subtitle, SOURCE, "D02_02_清洗步骤剩余记录.png")
plt.show()

In [ ]:
# ---- 读回断言：条长 = 剩余行数，最后一步 = 清洗后行数 ----
widths = [b.get_width() for b in ax.containers[0]]
assert np.allclose(widths, remain_vals), "条长和清洗步骤表对不上"
assert int(widths[-1]) == len(clean)
assert quality.fmt_pct(dup_share) in fig.texts[0].get_text()
print("读回断言通过：条长 = 剩余行数，末尾 = 清洗后行数")

## 重复记录：不是噪声，是批量写入

这是全项目影响最大的一项。31,994 行完全重复，去重后少 27%；
更关键的是重复**偏向取消单**，导致含重复口径把取消率从 27% 抬到 37%。

In [ ]:
# 重复的规模与构成：这一节回答"重复是不是随机噪声"
biz = config.BUSINESS_COLS
n_dup = int(raw.duplicated(subset=biz, keep="first").sum())
dedup = raw.drop_duplicates(subset=biz, keep="first")

rate_all = raw["is_canceled"].mean()          # 含重复口径
rate_ded = dedup["is_canceled"].mean()        # 去重口径
diff_pp = (rate_all - rate_ded) * 100

grp = raw.groupby(biz, dropna=False).size()   # 同一记录出现几次
bins = pd.cut(grp, [0, 1, 5, 20, np.inf], labels=["1 次", "2–5 次", "6–20 次", "21 次以上"])
size_dist = bins.value_counts().reindex(["1 次", "2–5 次", "6–20 次", "21 次以上"])

print(f"完全重复 {n_dup:,} 行（{quality.fmt_pct(n_dup / len(raw))}），去重后 {len(dedup):,} 行")
print(f"取消率：含重复 {quality.fmt_pct(rate_all)}，去重 {quality.fmt_pct(rate_ded)}，相差 {diff_pp:.0f} 个百分点")
size_dist

### 图 D02_03：重复记录对取消率的影响

In [ ]:
# ── 出图 D02_03：重复记录对取消率的影响 ────────────────────────────────
# 为什么左边只画两根条：这一节要说的是"口径差 10 个百分点"，两根一样长的坐标轴比什么都清楚。
# 为什么右边画组规模：说明重复不是零星噪声——有一组出现了 180 次，是批量写入的形态。
# 规范落在哪几行：两图各自的纵/横轴都从 0 开始；只标数值不画图例。
fig, (axa, axb) = plt.subplots(1, 2, figsize=(11, 4.2),
                               width_ratios=[1, 1.25], layout="constrained")

rates = [rate_all, rate_ded]
labels = ["含重复\n（119,390 行）", "去重后\n（87,396 行）"]
axa.barh([0, 1], rates, color=[RED, MAIN], height=0.5)
for i, r in enumerate(rates):
    axa.text(r + 0.012, i, f"{quality.fmt_pct(r)}", va="center", fontsize=11,
             fontweight="bold", color=RED if i == 0 else MAIN)
axa.set_yticks([0, 1], labels, fontsize=10)
axa.invert_yaxis()
axa.set_xlim(0, max(rates) * 1.3)
axa.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
axa.set_xlabel("取消率")
axa.grid(axis="x", visible=True)
axa.grid(axis="y", visible=False)
axa.set_title("同一份数据的两个取消率", loc="left")

xs = np.arange(len(size_dist))
axb.bar(xs, size_dist.to_numpy(), color=MAIN, width=0.6)
for x, v in zip(xs, size_dist.to_numpy()):
    axb.text(x, v * 1.03, f"{int(v):,} 组", ha="center", va="bottom", fontsize=9.5, color=INK)
axb.set_xticks(xs, size_dist.index)
axb.set_ylim(0, size_dist.max() * 1.25)
axb.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")     # 千分位：80000 读起来费劲
axb.set_ylabel("记录组数（组）")
axb.set_xlabel("同一记录出现的次数")
axb.set_title(f"重复组规模（最大一组出现 {int(grp.max())} 次）", loc="left")

# ---- 标题、副标题 ----
title = f"含重复 {quality.fmt_pct(rate_all)}、去重 {quality.fmt_pct(rate_ded)}，同一指标差 {diff_pp:.0f} 个百分点"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = ("完全重复 = 32 个业务字段全部相同，与 booking_id 无关（它是入库自增的）\n"
            "重复组里已取消订单占比更高，所以含重复口径会系统性高估取消率；报数必须注明口径")
viz.finish(fig, title, subtitle, SOURCE, "D02_03_重复记录对取消率的影响.png")
plt.show()

In [ ]:
# ---- 读回断言：两根条的长度就是两个口径的取消率 ----
heights = [b.get_width() for b in axa.containers[0]]
assert np.allclose(heights, rates), "条长和两个取消率对不上"
bars = axb.containers[0]
assert np.allclose([b.get_height() for b in bars], size_dist.to_numpy()), "组规模柱高对不上"
assert quality.fmt_pct(rate_ded) in fig.texts[0].get_text()
print("读回断言通过：左图条长 = 两个口径的取消率，右图柱高 = 组规模分布")

## 三种口径并列

与其争论"哪个取消率才对"，不如让读者看到差异从哪来。三种口径都不删，但**每个指标旁必须注明用的是哪一种**。

In [ ]:
# 三种口径并列：报告表 6-1 就是这张表
views = pd.DataFrame([
    {"口径": "原始层", "行数": len(raw), "取消率": raw["is_canceled"].mean(),
     "适用场景": "还原订单流水、审计", "风险": "重复行放大数量类指标"},
    {"口径": "去重后", "行数": len(dedup), "取消率": dedup["is_canceled"].mean(),
     "适用场景": "比例类指标、取消模型训练", "风险": "可能误删真实的相同预订"},
    {"口径": "清洗后", "行数": len(clean), "取消率": clean["is_canceled"].mean(),
     "适用场景": "客单价、间夜与收益分析", "风险": "样本减少 28%，绝对量不可直接比较"},
])
views["取消率"] = views["取消率"].map(quality.fmt_pct)
views

## 固化成一条命令

上面每一步都在 notebook 里手算过一遍。流水线做的事就是把这些步骤按同一顺序再跑一遍，
并留下运行记录——数字对不上会立刻暴露。

In [ ]:
# 把上面这些步骤固化成一条命令：读原始层 → 清洗 → 写明细层 → 16 项稽核 → 记日志
from hotel_dq import pipeline

log = pipeline.run_day()
chk = pd.read_sql(
    "SELECT (SELECT COUNT(*) FROM dwd_bookings) AS dwd, "
    "(SELECT COUNT(*) FROM dq_check_result) AS dq, "
    "(SELECT COUNT(DISTINCT 批次) FROM etl_log) AS runs", con)

# 对账：流水线跑出来的数字必须和 notebook 手算的一致
assert chk.loc[0, "dwd"] == len(clean) == 85_585
assert chk.loc[0, "dq"] == len(res) == 16
assert chk.loc[0, "runs"] >= 1
print(f"对账通过：dwd_bookings {chk.loc[0, 'dwd']:,} 行，dq_check_result {chk.loc[0, 'dq']} 条，"
      f"etl_log 里有 {chk.loc[0, 'runs']} 次运行")

log[["步骤", "输入行数", "输出行数", "耗时(秒)"]]

## 这一本的结论

| 编号 | 发现 | 关联 | 状态 | 处理决定 |
| --- | --- | --- | --- | --- |
| P-01 | 完全重复 31,994 行（27%），且 63% 是取消单 | DQ-06 | 不通过 | 报数时双口径并列；确认重复成因 |
| P-02 | 未到店 1,207 行计入 `is_canceled` | DQ-08 | 警告 | 需要区分时改用 `reservation_status` |
| P-03 | `company_id` 缺失 94.31% | DQ-02 | 不通过 | 降级为"是否协议客户"标记 |
| P-04 | 房价 0 值 1,959 行、负值 1 行、极值 5,400 一行 | DQ-10 | 不通过 | 均价一律排除 0 值 |
| P-05 | 房型代码 A—P 无业务字典 | DQ-16 | 警告 | 暂不开展房型价值分析 |
| P-06 | 样本仅 2 家酒店，客源集中于葡萄牙 | DQ-14 | 不通过 | 结论限定适用范围，不外推行业 |

**能用来做什么**：这两家酒店在 2015-07 至 2017-08 的预订结构、渠道差异、取消行为；
按去重口径算取消率与均价；构建取消预测模型并做特征重要度。
**不能用来做什么**：推断整个行业；做 2015 与 2017 的整年同比；把 `deposit_type` 当作取消的原因变量
（它和取消是反向因果，直接入模会标签泄漏）。

In [ ]:
con.close()   # 用完关闭数据库连接